# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [11]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'connect-four page',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'outsmart page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'nebula project',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [19]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [14]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 18 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'brand/about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'models catalog', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets catalog', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces catalog', 'url': 'https://huggingface.co/spaces'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn hub', 'url': 'https://huggingface.co/learn'},
  {'type': 'docs hub', 'url': 'https://huggingface.co/docs'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'GitHub', 'url': 

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [20]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 7 relevant links
## Landing Page:

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy 

In [29]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """



brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [23]:
get_brochure_user_prompt("Ed Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 9 relevant links


'\nYou are looking at a company called: Ed Donner\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,\nacquired in 2021\n.\nI will happil

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("Ed Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


# Ed Donner – Innovating AI with a Human Touch

---

## About Ed Donner

Ed Donner is a passionate AI engineer, entrepreneur, and educator who thrives at the intersection of advanced technology and human potential. As the co-founder and CTO of **Nebula.io**, Ed leads efforts to revolutionize recruitment using cutting-edge Generative AI and machine learning. Ed’s mission is to help people **discover their potential and pursue their reason for being**—a vision inspired by the Japanese concept of *Ikigai*, aiming to unlock greater fulfillment and prosperity for both individuals and organizations.

With a rich background as the founder and CEO of AI startup *untapt* (acquired in 2021), Ed combines entrepreneurial insight with technical excellence. Outside work, Ed enjoys experimenting with large language models (LLMs), coding, and even dabbling in amateur electronic music.

---

## What We Do: Nebula.io

Nebula.io transforms hiring by helping recruiters source, understand, engage, and manage talent more effectively through AI-powered tools. Their patented AI matching model goes beyond keywords to connect candidates to roles **more accurately and faster than ever before**. This innovation empowers companies to build stronger teams while improving job satisfaction and engagement for candidates.

- AI-driven recruitment technology
- Advanced, patented matching algorithms
- Focus on meaningful, fulfilling career alignment
- Free to try platform for recruiters and candidates

---

## Ed Donner’s AI Curriculum

Ed is also an acclaimed AI educator with **Udemy courses boasting over 500,000 students across 194 countries**. Known for his engaging and accessible style, he has crafted a comprehensive AI curriculum that guides learners from novice coders to proficient AI engineers. His courses cover practical, hands-on skills like building AI agents and deploying production-ready ML systems.

Key topics include:
- AI Coding and Engineering
- Building Smart Agents with tools like n8n
- MLOps and deploying AI models in production
- Strategic learning paths for AI mastery

For those passionate about AI, his courses serve as a gateway to becoming skilled practitioners capable of leveraging LLMs and modern AI tech.

---

## Innovation & Culture

- **Driven by curiosity and passion:** Ed openly shares his enthusiasm for AI and coding, often diving deep into LLMs and emerging technology.
- **Impact-oriented:** The team’s work targets real-world challenges, focusing on human well-being through technology.
- **Open and collaborative:** Ed’s approachable teaching style and interactive platforms promote community learning and growth.
- **Lifelong learners:** Embracing experimentation and continuous improvement, from music to machine learning.

---

## Career Opportunities

At Nebula.io, the core belief is that work should be inspiring and fulfilling. The company is likely to seek talented, motivated professionals who want to contribute to transformative AI projects that help others find their dream jobs. If you’re passionate about AI, human-centered tech, and innovation, keeping an eye on Nebula.io’s career opportunities could be rewarding.

---

## Get in Touch

Interested in learning more or collaborating? Reach out to Ed Donner directly:

- **Email:** ed [at] edwarddonner [dot] com  
- **Website:** [www.edwarddonner.com](http://www.edwarddonner.com)  
- **Social:** [LinkedIn](https://www.linkedin.com/in/edwarddonner) | [Twitter](https://twitter.com/edwarddonner) | [Facebook](https://www.facebook.com/edwarddonner)

---

Ed Donner and Nebula.io are reshaping the future of AI-driven recruitment, education, and human fulfillment — one algorithm at a time. Join the journey to unlock potential and inspire possibility.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [27]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [30]:
stream_brochure("Ed Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


# Welcome to Ed Donner's World: The AI Frontier with a Human Touch

---

## Who is Ed?

Meet Ed — not your average coder, he’s a code maestro and LLM (Large Language Model) whisperer with a penchant for *very* amateur electronic music production and profound nodding at Hacker News posts he barely understands. When he’s not blabbering about the mysteries of AI for hours on end (true story), he’s co-founded **Nebula.io**, the rocketship applying AI to something truly meaningful: *helping people discover their purpose* (yes, that deep).

---

## What’s Ed Up To?

- **Co-founder & CTO of Nebula.io**: Revolutionizing hiring by using patented AI models to match talent and roles — faster than you can say “404 keyword not found.” It’s free to try, no strings attached.
- **AI Startup Veteran**: Founded and led `untapt` until its smashing 2021 acquisition.
- **Udemy AI Course Creator**: After friends staged an intervention on his long AI monologues, Ed made courses. The result? Over 500,000 students from *194 countries* have enrolled. Yep, he’s basically an AI rockstar.
- **Creator of "Outsmart"**: An arena where LLMs battle for supremacy through diplomacy and deviousness. Think chess, but with AI brains and a hint of political intrigue.

---

## What’s Nebula.io?

Imagine if AI was your career counselor, fortune teller, and best friend — all rolled into one. Nebula.io taps Generative AI to:

- Help recruiters **source, understand, engage, and manage** talent with near-magical speed and precision.
- Use a patented model that finds the perfect job fit **without keywords** (because who really searches for jobs with keywords anymore?).
- Promote the ancient Japanese concept of **Ikigai** — helping people find fulfilling work that feeds their soul and prospers humanity.

With 77% of people feeling uninspired at work, Nebula.io is on a mission to raise workplace happiness worldwide. It’s serious business with a sunny side.

---

## Company Culture: Nerdy, Passionate, and Purpose-Driven

- **AI Obsession Welcome**: If you geek out over LLMs, pipelines, and MLOps, you’ll fit right in.
- **Continuous Learners**: Inspired by Ed’s Udemy courses, everyone grows — just like AI models connecting neurons.
- **Humor and Humanity**: Expect wit, occasional electronic beats, and a culture that values delight as much as data.
- **Impact First**: Working here means chasing not just technological breakthroughs but *human breakthroughs*.

---

## For Prospective Customers

- *Want smarter, faster hiring with maximum job satisfaction?* Nebula.io gives recruiters AI superpowers. No more keyword black holes; just perfect matches that stick.
- *Curious about AI’s real-world magic?* Check out “Outsmart” where AI brains negotiate, conspire, and outwit each other. Fun for you, revealing for your data scientists.

---

## For Investors

- Investing in Ed’s vision means backing AI that does *good* — lifting hiring headaches, increasing workplace engagement, and spreading Ikigai vibes worldwide.
- With a founder who’s a proven startup CEO and an AI influencer teaching half a million students, the future is bright, bold, and brilliantly coded.

---

## For Future Team Members

Ed's crew isn’t looking for just coders; they want AI dreamers, puzzle solvers, and impact seekers. If you:

- Love tinkering with LLMs and crafting AI agents (voice agents included),
- Can handle impromptu lectures about AI’s possibilities (resistance is futile),
- Want to combine tech with purpose and make real-world waves,

…then pack your digital bags and join the Nebula.io mission.

---

## Get in Touch

Feeling inspired? Reach out to Ed directly at  
✉️ ed [at] edwarddonner [dot] com

Or connect on your favorite social channels:  
[LinkedIn](https://linkedin.com/in/edwarddonner) | [Twitter](https://twitter.com/edwarddonner) | [Facebook](https://facebook.com/edwarddonner)

---

## Parting Words

In Ed’s own words:  
> “I’m in the business of finding people their dream jobs — and conveniently, I’m in my dream job myself."

Join us and let’s build the future of AI *and* human potential — one inspired hire at a time.

---

*Ed Donner & Nebula.io*  
Where AI’s brain meets the heart.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>